# Gerador de PDFs Premium (Medhelp)
Este notebook varre a pasta `Resumos_Prontos`, lê os arquivos Markdown gerados pelo Apps Script, realiza correções de listas e formatações, e gera PDFs com visualização premium (WeasyPrint).

**Recursos:**
- Correção automática de listas soltas e indentações de bullets
- Estilização de fluxos/ciclos (`A -> B -> C`) como chips conectados por setas
- Pula automaticamente resumos que já possuem PDF gerado

In [ ]:
!pip install -q weasyprint markdown
!apt-get install -y -q pango1.0-tools fonts-roboto

In [ ]:
import os
import re
import shutil
import time
import markdown
from weasyprint import HTML, CSS
from google.colab import drive

# Conectar Google Drive
drive.mount("/content/drive")

RESUMOS_DIR = "/content/drive/MyDrive/Logística - Drive/Transcrições/Resumos_Prontos"
PDFS_DIR = "/content/drive/MyDrive/Logística - Drive/Transcrições/PDFs_Premium"

ARQUIVO_DIR = "/content/drive/MyDrive/Logística - Drive/Transcrições/Resumos_Prontos/Arquivados"

os.makedirs(PDFS_DIR, exist_ok=True)
os.makedirs(ARQUIVO_DIR, exist_ok=True)

In [ ]:
CSS_PREMIUM = """
@page {
    size: A4;
    margin: 18mm 15mm 18mm 15mm;
    @bottom-center {
        content: counter(page);
        font-family: serif;
        font-size: 9pt;
        color: #94a3b8;
    }
    @bottom-left {
        content: "© Conteúdo Autoral • João Gabriel R. Trovão";
        font-family: serif;
        font-size: 8.5pt;
        color: #64748b;
    }
}

body {
    font-family: "Roboto", sans-serif;
    font-size: 10.5pt;
    line-height: 1.65;
    color: #2c3e50; /* Slate escuro orgânico */
    background-color: #ffffff;
}

/* ---- TÍTULOS E CABEÇALHOS (MIX EDITORIAL + CLEAN) ---- */
h1, h2, h3, h4 {
    font-family: "Roboto", serif; /* Serif para dar ar acadêmico/editorial */
    page-break-after: avoid;
}

h1 {
    font-size: 20pt;
    font-weight: 800;
    color: #1a1a1a;
    letter-spacing: -0.5px;
    padding-bottom: 8pt;
    margin-top: 0;
    margin-bottom: 1.2em;
    border-bottom: 1pt solid #cbd5e1; /* Hairline rule da Opção A */
}

h2 {
    font-size: 14pt;
    font-weight: 600;
    color: #556B2F; /* Verde Musgo */
    padding-bottom: 6pt;
    border-bottom: 0.5pt solid #e2e8f0; /* Linha de base limpa */
    margin-top: 2em;
    margin-bottom: 1em;
}

h3 {
    font-size: 11.5pt;
    font-weight: 700;
    color: #2c3e50;
    margin-top: 1.5em;
    margin-bottom: 0.5em;
}

strong {
    color: #1a1a1a;
    font-weight: 700;
}

/* Mistura B: Fundo pastel hiper-suave com borda da Opção A */
blockquote {
    border-left: 2pt solid #A3B18A; /* Verde Claro */
    margin: 1.5em 0;
    padding: 10pt 14pt;
    background-color: #fcfdfc; /* Fundo off-white areia/verde */
    color: #555555;
    font-style: italic;
    page-break-inside: avoid;
    border-radius: 0 4pt 4pt 0;
}

/* ---- TABELAS ---- */
table {
    width: 100%;
    border-collapse: collapse;
    margin: 2em 0;
    font-size: 9.5pt;
    page-break-inside: auto;
}

tr {
    page-break-inside: avoid;
}

th {
    color: #1a1a1a;
    padding: 8pt 0;
    text-align: left;
    font-weight: 700;
    border-bottom: 1pt solid #1a1a1a;
}

td {
    padding: 8pt 0;
    border-bottom: 0.5pt solid #e2e8f0;
    vertical-align: top;
}

/* ---- LISTAS E BULLETS NATIVOS ---- */
ul {
    padding-left: 18pt;
    margin: 1em 0;
    list-style-type: disc;
}

ul li::marker {
    color: #556B2F; /* Verde Musgo */
}

ul ul {
    padding-left: 16pt;
    margin: 4pt 0 4pt 0;
    list-style-type: circle;
}

ul ul li::marker {
    color: #A3B18A;
}

ol {
    padding-left: 18pt;
    margin: 1em 0;
}

li {
    margin-bottom: 6pt;
    line-height: 1.55;
}

/* ---- FLUXOS E CASCATAS (A -> B -> C) Mix ---- */
/* Opção B: Continua como "chips" de apostila, mas muito mais elegante */
.fluxo {
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    gap: 8pt;
    margin: 1em 0;
    page-break-inside: avoid;
}

.passo {
    background-color: #ffffff;
    border: 0.5pt solid #A3B18A; /* Borda Verde Claro */
    border-radius: 12pt;
    padding: 4pt 10pt;
    font-size: 9pt;
    color: #556B2F; /* Verde Musgo */
    font-weight: 600;
}

.seta {
    color: #A3B18A;
    font-weight: 800;
    font-size: 11pt;
}
"""

def normalizar_indentacao_listas(md_texto):
    """Garante 4 espaços de recuo para sub-itens de lista."""
    padrao = re.compile(r'^( +)([-*+]|\d+\.)(\s+)', re.MULTILINE)
    def corrigir(m):
        espacos, marcador, resto = m.groups()
        novos_espacos = " " * (len(espacos) * 2) if len(espacos) < 4 else espacos
        return f'{novos_espacos}{marcador}{resto}'
    return padrao.sub(corrigir, md_texto)

def corrigir_listas_soltas(md_texto):
    """Insere linha em branco obrigatória antes de qualquer lista grudada no parágrafo."""
    linhas = md_texto.split('\n')
    padrao_item = re.compile(r'^\s*([-*+]|\d+\.)\s+')
    saida = []
    dentro_de_lista = False

    for linha in linhas:
        eh_item = bool(padrao_item.match(linha))
        linha_vazia = linha.strip() == ''

        if eh_item and not dentro_de_lista and saida and saida[-1].strip() != '':
            saida.append('')

        saida.append(linha)

        if eh_item:
            dentro_de_lista = True
        elif not linha_vazia:
            dentro_de_lista = False

    return '\n'.join(saida)

def estilizar_fluxos_html(html):
    """Transforma sequências de setas ('A -> B -> C') em chips visuais."""
    padrao_fluxo = re.compile(
        r'<p>((?:(?!</p>).)*?-&gt;(?:(?!</p>).)*?)</p>',
        re.DOTALL
    )

    def substituir(m):
        conteudo = m.group(1)
        passos = [p.strip() for p in conteudo.split('-&gt;')]
        if len(passos) < 2:
            return m.group(0)
        miolo = '<span class="seta">→</span>'.join(
            f'<span class="passo">{p}</span>' for p in passos
        )
        return f'<div class="fluxo">{miolo}</div>'

    return padrao_fluxo.sub(substituir, html)

def gerar_pdf_de_markdown(filepath, pdf_path, css_string):
    with open(filepath, "r", encoding="utf-8") as f:
        md_content = f.read()

    # 1. Tratamento e Correção do Markdown
    md_content = normalizar_indentacao_listas(md_content)
    md_content = corrigir_listas_soltas(md_content)

    # 2. Conversão para HTML com extensões sane_lists, tables e fenced_code
    html_body = markdown.markdown(
        md_content,
        extensions=["tables", "fenced_code", "sane_lists"]
    )

    # 3. Pós-processamento visual de fluxos
    html_body = estilizar_fluxos_html(html_body)

    final_html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="utf-8">
</head>
<body>
    {html_body}
</body>
</html>"""

    # 4. Renderização do PDF via WeasyPrint
    HTML(string=final_html).write_pdf(
        pdf_path,
        stylesheets=[CSS(string=css_string)]
    )

def mover_arquivo_com_retry(origem, destino, max_retries=3):
    """Move um arquivo no Google Drive com Exponential Backoff para evitar I/O Errors (FUSE)."""
    for tentativa in range(max_retries):
        try:
            shutil.move(origem, destino)
            return True
        except Exception as e:
            print(f"⚠️ Erro ao mover {origem} (Tentativa {tentativa+1}/{max_retries}): {e}")
            time.sleep(2 ** tentativa) # Exponential backoff: 1s, 2s, 4s...
    print(f"❌ Falha definitiva ao mover {origem} após {max_retries} tentativas.")
    return False

def processar_diretorio(resumos_dir, pdfs_dir, arquivo_dir, css_string):
    processados = 0
    ignorados = 0

    for filename in os.listdir(resumos_dir):
        if filename.endswith(".md"):
            pdf_filename = filename.replace(".md", ".pdf")
            pdf_path = os.path.join(pdfs_dir, pdf_filename)
            filepath = os.path.join(resumos_dir, filename)
            arquivo_path = os.path.join(arquivo_dir, filename)

            if os.path.exists(pdf_path):
                ignorados += 1
                # Move o .md para a pasta de arquivos se o PDF já existir
                mover_arquivo_com_retry(filepath, arquivo_path)
                continue

            
            try:
                gerar_pdf_de_markdown(filepath, pdf_path, css_string)
            except Exception as e:
                print(f"❌ Erro fatal FUSE/Drive ao gerar PDF de {filename}: {e}")
                continue
            
            # Move o .md para a pasta de arquivos após gerar o PDF
            shutil.move(filepath, arquivo_path)
            
            print(f"✅ PDF gerado com sucesso: {pdf_filename}")
            processados += 1

    if processados == 0:
        print(f"Nenhum arquivo .md NOVO encontrado. ({ignorados} arquivos arquivados/ignorados).")
    else:
        print(f"\n🎉 Sucesso! {processados} PDFs gerados com qualidade premium. ({ignorados} antigos arquivados).")

if __name__ == '__main__':
    processar_diretorio(RESUMOS_DIR, PDFS_DIR, ARQUIVO_DIR, CSS_PREMIUM)
